***IMPORT THE NECESSARY MODULES***

In [1]:
# Cell 1: Setup autoreload
%load_ext autoreload
%autoreload 2

In [2]:
# Import the module that contains fucntions to read, open and write the EXCEL Table with the printer Fleet
import printers_data_ops

# Stands for "regular expression"; used for advanced string searching and manipulation. Thanks to this module we can extract easily chunks of strings 
# and place them on a separate column as column values (for example, the Base Code column)
import re

# A powerful library for data manipulation and analysis, primarily using DataFrames. We will use it to analyse and operate with the printer's data that
# has been imported from the dictionary.
import pandas as pd

# The module with the code to generate a printer report in HTML
import printer_report_craft

# The module with the code to generate data visualizations based on the printer data
import printer_plots

# This module provides the network operations to check the status of the printers
import printer_network_checks

We need to warn the user before continuing running the program to close the EXCEL file, if open:

In [15]:
# --- NEW: User Warning Section ---
print("===================================================================")
print(" *** IMPORTANT WARNING ***")
print(" Please ensure the Excel file 'Printer_Fleet_Table.xlsx' is CLOSED.")
print(" Keeping the file open will cause a 'Permission Denied' error.")
print("===================================================================")
input("Press Enter to continue...")
# -----------------------------

 *** IMPORTANT WARNING ***
 Please ensure the Excel file 'Printer_Fleet_Table.xlsx' is CLOSED.
 Keeping the file open will cause a 'Permission Denied' error.


Press Enter to continue... 


''


***DEFINE THE DATAFRAME***

First, we will define a DataFrame with pandas that contains all the dictionary keys as columns, so that we can operate and analize the data easily later on. We can enter whatever sheet we want to perform the check in: each sheet represents a site / excercise with its correspondent printers.

In [3]:
excel_sheet_name = input("Enter the EXCEL sheet name: ")
all_printers_df = printers_data_ops.create_printers_dataframe(excel_sheet_name)

Enter the EXCEL sheet name:  MRF-D


***CHECK ONLINE AND OFFLINE PRINTERS***

We will check which printers are online and which are offline. We will use pandas add a Online or Offline value to the Status column for each one.

In [4]:
printer_network_checks.printer_ping_check(all_printers_df, "Status", "Hostname", "IP Address")

Current progress: 100%

We can also check how the DataFrame looks like now, with the Status column added...

In [5]:
all_printers_df.head(2)

,Hostname,IP Address,Serial Number,Configuration,Status,Special Notes
0,ROBO-BL0252-FL1-RM001-LXMCX942,214.43.185.87,75800HK700944,Done,Online,NaN
1,ROBO-BL0252-FL1-RM002-LXMCX942,214.43.185.86,75800HK700912,Undone,Online,The printer has been reconnected to the MCEN n...


Lets add the Status column to the original EXCEL Table file, so the users can also see each printer state on the tracker.

In [6]:
printers_data_ops.update_excel_table_status(excel_sheet_name, all_printers_df)


Opening Excel file to update it while preserving formatting...
Successfully updated 'Printer_Fleet_Table.xlsx' while preserving its format.
Workbook closed successfully.


***GET A COUNT OF TOTAL ONLINE AND OFFLINE PRINTERS***

We will add a column called 'Base Code' to group printers by base later

In [7]:
all_printers_df['Base Code'] = all_printers_df['Hostname'].str[:4]

Lets start by grouping the printers by status

In [8]:
# Group rows by only the 'base_code' (site)
grouped_by_site = all_printers_df.groupby('Base Code')

Lets give the counts in text format...

In [9]:
# Get the count of each status
status_counts =  all_printers_df.groupby('Status')

# Display the text counts
print(status_counts.size())

Status
Offline    10
Online     21
dtype: int64


And also in a nice Pie Chart:

In [10]:
pie_chart_file_name = printer_plots.generate_printer_pie_chart(all_printers_df, "Status")

Pie chart successfully saved as 'printer_status_pie-chart.png'


***GET A COUNT OF TOTAL ONLINE AND OFFLINE PRINTERS PER BASE***

First, let's group the printers by base and status

In [11]:
# Group rows by the 'Base' and 'Status' value
grouped_by_base_and_status = all_printers_df.groupby(['Base Code', 'Status']).size()

# For a cleaner DataFrame format, you can add .reset_index()
grouped_by_base_and_status_df = grouped_by_base_and_status.reset_index(name='Count')

Now we can display the text counts:

In [12]:
# --- Display the results ---
print(grouped_by_base_and_status)

Base Code  Status 
DEBD       Online      1
LKBD       Offline     1
           Online      4
RAFD       Offline     2
           Online      2
ROBO       Offline     7
           Online     14
dtype: int64


Finally, we can use a bar cart to represent the Online and Offline printer per base

In [13]:
bar_chart_file_name = printer_plots.generate_printer_bar_chart(all_printers_df, "Base Code", "Status")

Bar chart with labels successfully saved as 'printer_status_by_base_code_bar-chart.png'


***GENERATE A PRINTER STATUS REPORT***

In [14]:
printer_report_craft.generate_printer_report_in_html(all_printers_df, pie_chart_file_name, bar_chart_file_name)

Report successfully generated and saved as 'Weekly_Printer_Report.html'
